# Mini-Project MP03 — Press Release to Plot

## Industry Comparison: Financial Services and Travel and Hospitality

*CIS 3120 — Programming for Analytics*
*Baruch College, Zicklin School of Business*

---

**Team number:** `<02>` (replace with two-digit number from Brightspace)

**Team members:**
- Financial Services Pipeline Lead: `<Michael Gartsbeyn>`
- Travel and Hospitality Pipeline Lead: `<Andy Huang>`
- Comparison and Visualization Lead (Integrator): `<Raghav Sharma>`

**Submission filename:** `MP03_Notebook_team_<02>.ipynb`

---

## How to use this starter

1. Make a copy of this notebook and rename it `MP03_Notebook_team_<NN>.ipynb` using your team number.
2. Replace the User-Agent placeholder in the setup cell with your Baruch email.
3. Configure your Anthropic API key in Colab Secrets as `ANTHROPIC_API_KEY`.
4. Work through the notebook section by section. Sections marked **CANONICAL** are the validated Module 15 pipeline and must not be modified. Sections marked **TODO** are where your team writes new code.
5. Run the window-tuning experiment, populate the results table, build the integrated map, and complete the methodology and reflection sections.
6. Verify the notebook runs end-to-end (Runtime → Restart and run all in Colab) before submitting.

See `docs/MP03_Assignment.docx` for the full assignment specification.

---

## 1. Setup

Install dependencies (Colab) and configure the request headers and API client.

In [ ]:
# Colab installs (silent). The other packages are pre-installed in the Colab base image.
!pip install anthropic folium --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 9.9 MB/s eta 0:00:00


In [ ]:
import json
import re
import time
from datetime import date, datetime, timedelta

import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic

# ─────────────────────────────────────────────────────────────────────────
# CRITICAL: Replace the placeholder below with your Baruch email.
# Both SEC EDGAR and OpenStreetMap Nominatim require a descriptive
# User-Agent header. Generic agents are rejected with HTTP 403.
# ─────────────────────────────────────────────────────────────────────────
USER_AGENT = "CIS3120 MP03 Team <02> - raghav.sharma1@baruch.cuny.edu"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

# ─────────────────────────────────────────────────────────────────────────
# Endpoints and constants
# ─────────────────────────────────────────────────────────────────────────
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"

EDGAR_PAUSE      = 0.15   # seconds between EDGAR requests (SEC: 10 req/sec)
NOMINATIM_PAUSE  = 1.10   # seconds between Nominatim requests (1 req/sec)

# Anthropic model: current Haiku in the Claude 4.5 family.
MODEL_ID = "claude-haiku-4-5-20251001"

In [ ]:
# Configure the Anthropic API client from Colab Secrets.
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
client = Anthropic(api_key=ANTHROPIC_API_KEY)

In [ ]:
#Clone the course repo so mp03 package is available in Colab
!git clone -b mp/03-industry-comparison-team-02 https://github.com/R-Sharma101/cis3120-spring2026.git
import os
os.chdir('cis3120-spring2026')

Cloning into 'cis3120-spring2026'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 111 (delta 45), reused 38 (delta 34), pack-reused 56 (from 1)
Receiving objects: 100% (111/111), 122.51 KiB | 2.08 MiB/s, done.
Resolving deltas: 100% (49/49), done.


In [ ]:
# Import the seeded ticker lists and search-phrase lists from the mp03 module.
# If the mp03 package is not on the Python path, append the parent directory.
import sys
from pathlib import Path

# When running in Colab from a cloned repo, this places the repo root on sys.path.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mp03.seeds import (
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
)

print(f"Financial Services tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Financial Services phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Travel and Hospitality tickers: {len(TRAVEL_HOSPITALITY_TICKERS)}")
print(f"Travel and Hospitality phrases: {len(TRAVEL_HOSPITALITY_PHRASES)}")

Financial Services tickers: 33
Financial Services phrases: 14
Travel and Hospitality tickers: 16
Travel and Hospitality phrases: 11


---

## 2. Canonical Pipeline (Module 15)

The five functions in this section are the preserved pipeline from the Module 15 instructor notebook. **Do not modify these signatures.** Downstream code in this notebook calls them with these exact argument shapes.

### Stage 1 — Retrieve candidate 8-K filings from EDGAR

Each phrase is queried independently. Combining phrases with boolean OR inside parentheses is a documented but non-functional approach in the SEC's full-text search engine and must not be used.

In [ ]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window.

    Returns a tuple of (list of hit dicts, total reported by EDGAR).
    """
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q":         phrase,
            "dateRange": "custom",
            "startdt":   start_date.isoformat(),
            "enddt":     end_date.isoformat(),
            "forms":     forms,
            "from":      page * 100,
        }
        response = requests.get(
            EDGAR_SEARCH_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total

In [ ]:
def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 250,
) -> list[dict]:
    """Run search_edgar_one_phrase across a list of phrases with retry-with-backoff.

    Deduplicates by (accession number, exhibit filename). Stops accumulating
    once max_filings is reached.
    """
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]

    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(
                    phrase, start_date, end_date, forms, max_pages
                )
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: phrase {phrase!r} failed after retries ({exc}); skipping")
                    hits = []
                    break
                wait = backoff_waits[attempts]
                print(f"  transient error on {phrase!r}: {exc}. retrying in {wait}s...")
                time.sleep(wait)
                attempts += 1

        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)

    return deduped

### Stage 2 — Fetch the press release text from each filing

In [ ]:
def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit referenced by the hit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik}/{accession_no_dashes}/{filename}"
    )


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text for a single hit.

    Returns (text, url). Truncates at max_chars (~2000 tokens).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"
    return text, url

### Stage 3 — Classify and extract with the Anthropic API

The system prompt below achieved 100 percent precision in prototype testing. Use it verbatim.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing.

    Expects filing dict with keys: text (str), url (str), and any other
    metadata to be preserved on the returned record. Returns a dict
    extending filing with the parsed extraction fields and token usage.
    """
    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}

    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record

### Stage 4 — Geocode the locations

Nominatim enforces a strict 1-request-per-second policy. The 1.10-second pause is a comfortable margin.

In [ ]:
def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via OpenStreetMap Nominatim.

    Returns (latitude, longitude) on success, None if no match is found.
    """
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=REQUEST_HEADERS,
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])

### Stage 5 — Render the folium map (base configuration)

The base map and event color palette are provided. Your team will customize the marker rendering in Section 5 below to encode both industry and event type.

In [ ]:
EVENT_COLORS = {
    "opening":    "green",
    "closing":    "red",
    "relocation": "orange",
    "expansion":  "blue",
    "other":      "gray",
}

# Reasonable default center (geographic center of the contiguous US).
US_CENTER_LAT = 39.8
US_CENTER_LON = -98.6

---

## 3. Required New Functions (TODO)

Each team adds the three functions below. Each one has a single, well-defined responsibility. Do not bundle multiple responsibilities into one function.

Reference: `docs/MP03_Assignment.docx`, Section 3.

In [ ]:
def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """Restrict a candidate set returned by Stage 1 to a list of tickers.

    Implementation hint: each EDGAR hit has hit["_source"]["tickers"];
    match case-insensitively and return only matching hits.

    Parameters
    ----------
    candidates : list[dict]
        EDGAR hits as returned by search_edgar_all_phrases.
    ticker_list : list[str]
        Tickers to retain (e.g., FINANCIAL_SERVICES_TICKERS).

    Returns
    -------
    list[dict]
        The subset of candidates whose tickers intersect ticker_list.
    """

    ticker_set = {t.upper() for t in ticker_list}
    filtered = []
    for hit in candidates:
        display_names = hit['_source'].get('display_names', [])
        for name in display_names:
            matches = re.findall(r'\(\s*([A-Z]{1,5})\s*\)', name)
            if any(m in ticker_set for m in matches):
                filtered.append(hit)
                break
    return filtered

In [ ]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> list[dict]:
    """Run all five pipeline stages for one industry slice.

    Returns geocoded events with an "industry" field added to each record.

    Parameters
    ----------
    industry_label : str
        Either "Financial Services" or "Travel and Hospitality".
    ticker_list : list[str]
        Industry-specific ticker list.
    phrase_list : list[str]
        Industry-specific search-phrase list.
    window_days : int
        Length of the EDGAR date window (e.g., 30, 60, 90, 180, 360).

    Returns
    -------
    list[dict]
        One dict per geocoded location event, with the "industry" field set
        to industry_label. Records that fail classification or geocoding are
        excluded from the return value.

    Implementation guidance
    -----------------------
    1. Compute start_date and end_date from window_days.
    2. Call search_edgar_all_phrases(phrase_list, start_date, end_date).
    3. Filter the candidate list with filter_candidates_by_tickers.
    4. For each filtered candidate: fetch_exhibit_text, then extract_with_claude.
    5. Keep only records where is_location_event is True.
    6. For each kept record, geocode via geocode_location; drop records that
       fail geocoding.
    7. Add the "industry" field to each surviving record.
    """
    end_date = date.today()
    start_date = end_date - timedelta(days=window_days)

    print(f"\n--- {industry_label} | window={window_days} days ---")

    # Stage 1: search EDGAR
    candidates = search_edgar_all_phrases(phrase_list, start_date, end_date)
    print(f"  Stage 1: {len(candidates)} candidates before filtering")

    # Stage 2: filter by tickers
    candidates = filter_candidates_by_tickers(candidates, ticker_list)
    print(f"  After ticker filter: {len(candidates)} candidates")

    events = []
    for hit in candidates:
        # Stage 2: fetch text
        try:
            text, url = fetch_exhibit_text(hit)
        except Exception as e:
            print(f"  fetch error: {e}")
            continue

        filing = {**hit["_source"], "text": text, "url": url}

        # Stage 3: classify with Claude
        record = extract_with_claude(filing)
        if not record.get("is_location_event"):
            continue

        # Stage 4: geocode
        coords = geocode_location(record.get("city"), record.get("state"))
        if coords is None:
            continue
        record["lat"], record["lon"] = coords
        record["industry"] = industry_label
        events.append(record)

    print(f"  Final: {len(events)} location events")
    return events

In [ ]:
def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record the result of one window-tuning trial.

    Returns a dict that is directly appendable to the window-experiment
    results table.

    Parameters
    ----------
    industry_label : str
        Either "Financial Services" or "Travel and Hospitality".
    window_days : int
        One of 30, 60, 90, 180, 360.
    candidate_count : int
        Length of filtered candidate list before Stage 3.
    event_count : int
        Number of records where is_location_event is True.
    estimated_cost_usd : float
        Approximate API spend for this trial; sum of input + output token
        cost at Haiku 4.5 pricing ($1/M input, $5/M output).

    Returns
    -------
    dict
        Row with keys: industry, window_days, candidate_count, event_count,
        estimated_cost_usd.
    """
    return {
        "industry": industry_label,
        "window_days": window_days,
        "candidate_count": candidate_count,
        "event_count": event_count,
        "estimated_cost_usd": round(estimated_cost_usd, 4),
    }

---

## 4. Window-Tuning Experiment

Determine the smallest window that produces at least 8 location events for both industries without exceeding the $3.00 cumulative cost ceiling.

**Protocol:**
1. Begin at `WINDOW_DAYS = 30`. Run the pipeline for both industries.
2. If both industries reach the event-count target, stop.
3. Otherwise advance through 60, 90, 180, 360. Stop at the first window where both industries reach the target, or at 360, whichever comes first.

**Stopping criteria:**

| Criterion | Threshold |
|:---|:---|
| Event-count target | At least 8 location events per industry |
| Cost ceiling | $3.00 cumulative across all trials |
| Window ceiling | 360 days |

Reference: `docs/MP03_Assignment.docx`, Section 4.

In [ ]:
# Initialize the window-experiment results table.
# Append one row per (industry, window) trial that you actually run.
window_results = pd.DataFrame(columns=[
    "industry",
    "window_days",
    "candidate_count",
    "event_count",
    "estimated_cost_usd",
])

window_results

,industry,window_days,candidate_count,event_count,estimated_cost_usd


### 4.1 Window trials — Financial Services

Run the pipeline for Financial Services at successive window lengths and append a row to `window_results` after each trial using `summarize_window_trial`.

In [ ]:
fs_events_30 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=30,
)
fs_cost_30 = sum(
    r.get("input_tokens", 0) * 0.000001 +
    r.get("output_tokens", 0) * 0.000005
    for r in fs_events_30
)
fs_row = summarize_window_trial(
    industry_label="Financial Services",
    window_days=30,
    candidate_count=10,
    event_count=len(fs_events_30),
    estimated_cost_usd=fs_cost_30,
)
window_results = pd.concat([window_results, pd.DataFrame([fs_row])], ignore_index=True)
window_results


--- Financial Services | window=30 days ---
  Stage 1: 250 candidates before filtering
  After ticker filter: 10 candidates
  Final: 3 location events


/tmp/ipykernel_19402/322917878.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  window_results = pd.concat([window_results, pd.DataFrame([fs_row])], ignore_index=True)


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,10,3,0.0086


In [ ]:
fs_events_60 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=60,
)
fs_cost_60 = sum(
    r.get("input_tokens", 0) * 0.000001 +
    r.get("output_tokens", 0) * 0.000005
    for r in fs_events_60
)
window_results = pd.concat([window_results, pd.DataFrame([summarize_window_trial(
    industry_label="Financial Services",
    window_days=60,
    candidate_count=11,
    event_count=len(fs_events_60),
    estimated_cost_usd=fs_cost_60,
)])], ignore_index=True)
window_results


--- Financial Services | window=60 days ---
  Stage 1: 250 candidates before filtering
  After ticker filter: 11 candidates
  Final: 3 location events


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,10,3,0.0086
1,Financial Services,60,11,3,0.0086


In [ ]:
fs_events_90 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=90,
)
fs_cost_90 = sum(
    r.get("input_tokens", 0) * 0.000001 +
    r.get("output_tokens", 0) * 0.000005
    for r in fs_events_90
)
window_results = pd.concat([window_results, pd.DataFrame([summarize_window_trial(
    industry_label="Financial Services",
    window_days=90,
    candidate_count=13,
    event_count=len(fs_events_90),
    estimated_cost_usd=fs_cost_90,
)])], ignore_index=True)
window_results


--- Financial Services | window=90 days ---
  Stage 1: 250 candidates before filtering
  After ticker filter: 13 candidates
  Final: 3 location events


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,10,3,0.0086
1,Financial Services,60,11,3,0.0086
2,Financial Services,90,13,3,0.0086


In [ ]:
fs_events_180 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=180,
)
fs_cost_180 = sum(
    r.get("input_tokens", 0) * 0.000001 +
    r.get("output_tokens", 0) * 0.000005
    for r in fs_events_180
)
window_results = pd.concat([window_results, pd.DataFrame([summarize_window_trial(
    industry_label="Financial Services",
    window_days=180,
    candidate_count=28,
    event_count=len(fs_events_180),
    estimated_cost_usd=fs_cost_180,
)])], ignore_index=True)
window_results


--- Financial Services | window=180 days ---
  Stage 1: 250 candidates before filtering
  After ticker filter: 28 candidates
  Final: 6 location events


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,10,3,0.0086
1,Financial Services,60,11,3,0.0086
2,Financial Services,90,13,3,0.0086
3,Financial Services,180,28,6,0.0147


In [ ]:
fs_events_360 = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=360,
)
fs_cost_360 = sum(
    r.get("input_tokens", 0) * 0.000001 +
    r.get("output_tokens", 0) * 0.000005
    for r in fs_events_360
)
window_results = pd.concat([window_results, pd.DataFrame([summarize_window_trial(
    industry_label="Financial Services",
    window_days=360,
    candidate_count=40,
    event_count=len(fs_events_360),
    estimated_cost_usd=fs_cost_360,
)])], ignore_index=True)
window_results


--- Financial Services | window=360 days ---
  Stage 1: 250 candidates before filtering
  After ticker filter: 40 candidates
  Final: 10 location events


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,10,3,0.0086
1,Financial Services,60,11,3,0.0086
2,Financial Services,90,13,3,0.0086
3,Financial Services,180,28,6,0.0147
4,Financial Services,360,40,10,0.0256


### 4.2 Window trials — Travel and Hospitality

In [ ]:
th_events_30 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=30,
)
th_cost_30 = sum(
    r.get("input_tokens", 0) * 1e-6 +
    r.get("output_tokens", 0) * 5e-6
    for r in th_events_30
)
th_row = summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=30,
    candidate_count=len(th_events_30),
    event_count=len(th_events_30),
    estimated_cost_usd=th_cost_30,
)
window_results = pd.concat([window_results, pd.DataFrame([th_row])], ignore_index=True)
window_results


--- Travel and Hospitality | window=30 days ---
  Stage 1: 32 candidates before filtering
  After ticker filter: 0 candidates
  Final: 0 location events


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,10,3,0.0086
1,Financial Services,60,11,3,0.0086
2,Financial Services,90,13,3,0.0086
3,Financial Services,180,28,6,0.0147
4,Financial Services,360,40,10,0.0256
5,Travel and Hospitality,30,0,0,0.0000


In [ ]:
# 60-day trial — Travel and Hospitality
th_events_60 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=60,
)
th_cost_60 = sum(
    r.get("input_tokens", 0) * 1e-6 +
    r.get("output_tokens", 0) * 5e-6
    for r in th_events_60
)
window_results = pd.concat([window_results, pd.DataFrame([summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=60,
    candidate_count=len(th_events_60),
    event_count=len(th_events_60),
    estimated_cost_usd=th_cost_60,
)])], ignore_index=True)

window_results


--- Travel and Hospitality | window=60 days ---
  Stage 1: 44 candidates before filtering
  After ticker filter: 0 candidates
  Final: 0 location events


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,10,3,0.0086
1,Financial Services,60,11,3,0.0086
2,Financial Services,90,13,3,0.0086
3,Financial Services,180,28,6,0.0147
4,Financial Services,360,40,10,0.0256
5,Travel and Hospitality,30,0,0,0.0000
6,Travel and Hospitality,60,0,0,0.0000


In [ ]:
th_events_360 = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=360,
)
th_cost_360 = sum(
    r.get("input_tokens", 0) * 0.000001 +
    r.get("output_tokens", 0) * 0.000005
    for r in th_events_360
)
window_results = pd.concat([window_results, pd.DataFrame([summarize_window_trial(
    industry_label="Travel and Hospitality",
    window_days=360,
    candidate_count=len(th_events_360),
    event_count=len(th_events_360),
    estimated_cost_usd=th_cost_360,
)])], ignore_index=True)
window_results


--- Travel and Hospitality | window=360 days ---
  Stage 1: 250 candidates before filtering
  After ticker filter: 3 candidates
  Final: 0 location events


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,10,3,0.0086
1,Financial Services,60,11,3,0.0086
2,Financial Services,90,13,3,0.0086
3,Financial Services,180,28,6,0.0147
4,Financial Services,360,40,10,0.0256
5,Travel and Hospitality,30,0,0,0.0000
6,Travel and Hospitality,60,0,0,0.0000
7,Travel and Hospitality,360,0,0,0.0000


### 4.3 Selected window and final pipeline runs

Once both industries reach the event-count target at a common window length, record the chosen window below and run the final pipeline for both industries at that window. The events from these two final runs feed Section 5.

In [ ]:
CHOSEN_WINDOW_DAYS = 360

fs_events = run_industry_pipeline(
    "Financial Services",
    FINANCIAL_SERVICES_TICKERS,
    FINANCIAL_SERVICES_PHRASES,
    window_days=CHOSEN_WINDOW_DAYS,
)

th_events = run_industry_pipeline(
    "Travel and Hospitality",
    TRAVEL_HOSPITALITY_TICKERS,
    TRAVEL_HOSPITALITY_PHRASES,
    window_days=CHOSEN_WINDOW_DAYS,
)

all_events = fs_events + th_events
print(f"Financial Services:     {len(fs_events)} events")
print(f"Travel and Hospitality: {len(th_events)} events")
print(f"Total:                  {len(all_events)} events")


--- Financial Services | window=360 days ---
  Stage 1: 250 candidates before filtering
  After ticker filter: 40 candidates
  Final: 10 location events

--- Travel and Hospitality | window=360 days ---
  Stage 1: 250 candidates before filtering
  After ticker filter: 3 candidates
  Final: 0 location events
Financial Services:     10 events
Travel and Hospitality: 0 events
Total:                  10 events


---

## 5. Integrated Folium Map

Build a single map containing markers from both industries. The visual encoding must distinguish industry and event type **simultaneously and unambiguously**. The recommended scheme is:

- **Industry** by marker color family (e.g., navy for Financial Services, teal for Travel and Hospitality).
- **Event type** by marker icon shape (e.g., `home` for opening, `times-circle` for closing).

Each marker's popup must display: company name, ticker, industry label, filing date, event type, summary, and a working hyperlink to the underlying SEC filing.

Reference: `docs/MP03_Assignment.docx`, Section 7 (verification checklist).

In [ ]:
INDUSTRY_COLORS = {
    "Financial Services":     "darkblue",
    "Travel and Hospitality": "darkgreen",
}

EVENT_ICONS = {
    "opening":    "home",
    "closing":    "times-circle",
    "relocation": "exchange",
    "expansion":  "arrows-alt",
    "other":      "question-circle",
}

m = folium.Map(
    location=[US_CENTER_LAT, US_CENTER_LON],
    zoom_start=4,
    tiles="CartoDB positron",
)

for event in all_events:
    company    = event.get("entity_name") or event.get("company_name") or "Unknown Company"
    tickers    = event.get("tickers", [])
    ticker_str = ", ".join(tickers) if tickers else "N/A"
    industry   = event.get("industry", "Unknown")
    filed_at   = event.get("file_date") or event.get("period_of_report") or "Unknown"
    event_type = event.get("event_type") or "other"
    summary    = event.get("summary") or "No summary available."
    url        = event.get("url", "#")

    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 13px; width: 300px;">
      <b style="font-size:14px;">{company}</b><br>
      <span style="color:#555;">Ticker: {ticker_str}</span><br>
      <span style="color:#555;">Industry: {industry}</span><br>
      <span style="color:#555;">Filing date: {filed_at}</span><br>
      <span style="color:#555;">Event type: <b>{event_type.title()}</b></span><br>
      <hr style="margin:6px 0;">
      <i>{summary}</i><br>
      <a href="{url}" target="_blank" style="color:#1a73e8;">View SEC filing ↗</a>
    </div>
    """

    folium.Marker(
        location=[event["lat"], event["lon"]],
        popup=folium.Popup(popup_html, max_width=350),
        tooltip=f"{company} — {event_type.title()} ({industry})",
        icon=folium.Icon(
            color=INDUSTRY_COLORS.get(industry, "gray"),
            icon=EVENT_ICONS.get(event_type, "question-circle"),
            prefix="fa",
        ),
    ).add_to(m)

legend_html = """
<div style="position: fixed; bottom: 40px; left: 40px; z-index: 1000;
     background-color: white; padding: 14px 18px; border-radius: 8px;
     border: 1px solid #ccc; font-family: Arial, sans-serif; font-size: 13px;
     box-shadow: 2px 2px 6px rgba(0,0,0,0.2);">
  <b style="font-size:14px;">Legend</b><br><br>
  <b>Industry (marker color)</b><br>
  <span style="color:#00008B;">&#9679;</span> Financial Services<br>
  <span style="color:#006400;">&#9679;</span> Travel &amp; Hospitality<br>
  <br>
  <b>Event type (icon)</b><br>
  &#8962; Opening &nbsp;&nbsp; &#10007; Closing<br>
  &#8644; Relocation &nbsp;&nbsp; &#8615; Expansion<br>
  ? Other
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m

### Export the map to `maps/mp03_map_team_<NN>.html`

In [ ]:
import os
os.makedirs("../maps", exist_ok=True)
OUTPUT_PATH = "../maps/mp03_map_team_02.html"
m.save(OUTPUT_PATH)
print(f"Map saved to {OUTPUT_PATH}")

Map saved to ../maps/mp03_map_team_02.html


---

## 6. Methodology

The content below also appears as a standalone Markdown file at `methodology/mp03_methodology_team_<NN>.md`. Both copies must contain the same content; the standalone file is the version graded.

### 6.1 Ticker-list rationale

For the Financial Services pipeline, tickers such as FMCB, UNTY, and other
small regional banks were added to the seeded list to cover more geographic
areas and increase the likelihood of finding location events.

For Travel and Hospitality, JBLU and IHG were added to better represent
budget airlines and international hotels, which were undercounted in the
original seed.


### 6.2 Search-phrase rationale

For Financial Services, phrases such as "new office" and "branch relocation"
were added to capture location events that the original seed phrases missed.

For Travel and Hospitality, "new service" and "new destination" were added to
target airline route launches and new property announcements. Despite these
additions, the T&H phrase set did not produce candidates that survived ticker
filtering, suggesting T&H companies announce expansions through channels other
than SEC 8-K filings.


### 6.3 Window-experiment results

```
| Industry | Window (days) | Candidates | Location Events | Est. Cost (USD) |
|---|---|---|---|---|
| Financial Services | 30 | 10 | 3 | $0.0086 |
| Financial Services | 60 | 11 | 3 | $0.0086 |
| Financial Services | 90 | 13 | 3 | $0.0086 |
| Financial Services | 180 | 28 | 6 | $0.0147 |
| Financial Services | 360 | 40 | 10 | $0.0256 |
| Travel and Hospitality | 30 | 0 | 0 | $0.0000 |
| Travel and Hospitality | 60 | 0 | 0 | $0.0000 |
| Travel and Hospitality | 360 | 0 | 0 | $0.0000 |
```
We started with 5.00 USD in API credits and have 3.16 USD remaining, meaning we spent about 1.84 USD total including all development and trial runs. The actual window tuning trials cost well under 0.10 USD combined, which is way below the 3.00 USD ceiling. Haiku 4.5 pricing is 1.00 USD per million input tokens and 5.00 USD per million output tokens, so the low cost makes sense given how few candidates survived the ticker filter at each window.

We chose 360 days as our final window because Financial Services did not hit 8 events until that point. We ran every window from 30 to 360 days for FS and kept getting under 8 events until the last one. For T&H we got 0 events at every window we tried so we stopped at 360 since that is the ceiling. The T&H shortfall is explained further in sections 6.4 and 6.5.


### 6.4 Stage 3 classification quality per industry

*In the search 10 financial service location events were extracted and reviewed with the results from 360 days. For each of these 10 services the city and state as well as event type were filled with data. There were no false positives found in the data as each one was identified correctly as a new branch of the bank opening soon. Madison NJ and Medford MA seemed to appear multiple times showing various banks opening in the same areas. The summary showed data for each of the new banks opening and it seems that the quality for this data was good as there were 10 confirmed events after going through the 40 candidates from the filter showing a precision of 25%.*

At the 360 day window, stage 1 returned 250 candidates before ticker filtering. After applying the ticker, 3 candidates remained. Claude classified aall 3 as non-location events. the likely explanation is that the 3 filitered fillings mentioned T&H tickers in passing rather than announcing a specific physical facility opening, closing, relocation, or expansion. This is conistent with the strick classification criteria in the system prompt. The shortfall at 360 days means the T&H pipeline produced no geocoded events. This limits the comparative analysis and is addressed honestly in the reflection section.

### 6.5 Limitations

1. T&H had zero events. Even after running 360 days, only 3 candidates made
   it through the ticker filter and Claude classified all 3 as not location
   events. Because of this we could only map Financial Services data.

2. The ticker filter may have been too strict. It only matches companies
   whose ticker appears in EDGAR's display names field. A lot of T&H companies
   probably filed under different names so they got dropped before even
   reaching Claude.

3. Geocoding only works for US locations. We added USA to every search so
   anything international would not show up on the map.

4. Long filings get cut off at 8000 characters. If the location info was
   near the end of a filing it would have been missed entirely.

5. Cost estimates are not exact. We only counted tokens for candidates that
   passed the ticker filter. The ones that got rejected also used tokens so
   the real cost is a little higher than what we reported.


---

## 7. Comparative Reflection

A 300-to-400-word reflection on what the geographic patterns reveal about how the two industries deploy and consolidate physical capacity, and what the differences imply about each industry's underlying economics.

The same content appears as a standalone Markdown file at `reflections/mp03_reflection_team_<NN>.md`.

The integrated map shows 10 Financial Services location events spread across the
northeastern United States, with clusters in New Jersey and Massachusetts. The
events are almost entirely new branch openings, which reflects how retail banks
grow — by adding physical locations in densely populated areas where foot traffic
and local deposits justify the cost of a new branch.

The geographic concentration in the Northeast makes sense given that the tickers
in our list (JPM, BAC, WFC, PNC, USB, TFC) are heavily concentrated in that
region historically. Madison, NJ and Medford, MA appeared multiple times,
suggesting multiple banks are targeting the same suburban markets simultaneously.
This points to a competitive dynamic where banks track each other's expansion
decisions rather than independently choosing locations.

Travel and Hospitality produced zero location events at the 360-day window
ceiling, which prevented a direct geographic comparison. The most likely
explanation is that T&H companies disclose physical expansions differently than
banks do  hotel openings are often announced through press releases that do not
use the specific phrases we searched for, and airline route launches may appear
in earnings calls or investor presentations rather than 8-K exhibits. This is a
pipeline limitation rather than evidence that T&H companies are not expanding
physically.

Had T&H events been captured, we would have expected a very different geographic pattern hotel openings clustering in leisure destinations like Florida, Las Vegas, and coastal resort markets, while airline events would concentrate near major hub airports. The contrast between Financial Services expanding into dense suburban markets and Travel and Hospitality expanding into leisure and tourism corridors would have illustrated how each industry follows a fundamentally different demand signal. That comparison remains a direction for future work with a more targeted phrase set.

---

## 8. Pre-Submission Verification

Before the integrator submits, confirm each of the following:

- [ ] Notebook restarts cleanly and runs end-to-end (Runtime → Restart and run all in Colab).
- [ ] No committed API keys, no hard-coded credentials, no leftover debug prints.
- [ ] `window_results` table is populated with at least one row per (industry, window) trial actually run.
- [ ] Both industries reach at least 8 location events at the chosen window, OR a 360-day trial was run for both and the short-fall is acknowledged in Section 6.
- [ ] Cumulative window-tuning cost is at or below $3.00.
- [ ] Integrated map renders inline AND is exported to `maps/mp03_map_team_<NN>.html`.
- [ ] Every marker has a popup with all required fields and a working SEC hyperlink.
- [ ] Industry is visually distinguishable from event type on the map.
- [ ] Methodology appears both in this notebook and at `methodology/mp03_methodology_team_<NN>.md`.
- [ ] Comparative reflection appears both in this notebook and at `reflections/mp03_reflection_team_<NN>.md`.
- [ ] Team branch name is exactly `mp/03-industry-comparison-team-<NN>` and submission tag `mp03-team-<NN>` is pushed.
- [ ] At least three commits per team member following the `feat(scope): description` convention appear in the merged history.
- [ ] Brightspace submission text field contains the upstream PR URL and the names of all three team members with their roles.